# H-S00 MONAI SegResNet baseline

`H-S00.yaml`을 단일 설정으로 사용합니다. 학습 프로세스는 별도로 실행되며 진행률, ETA, Dice, Recall, composite, early stopping 및 GPU 메모리를 `progress.json`에서 읽어 표시합니다.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path(r'C:\model')
CONFIG_PATH = REPO_ROOT / 'configs/experiments/hemorrhage/H-S00.yaml'
sys.path.insert(0, str(REPO_ROOT / 'src'))

from brainon_ai.models.hemorrhage.segresnet.config import load_config, resolve_model_path
config = load_config(CONFIG_PATH)
RUN_DIR = resolve_model_path(config['experiment']['output_dir'])
PROGRESS_PATH = RUN_DIR / 'progress.json'
LOG_PATH = RUN_DIR.parent / 'H-S00-launch.log'

print('config:', CONFIG_PATH)
print('manifest:', config['experiment']['manifest_path'])
print('data root:', config['data']['root_dir'])
print('output:', RUN_DIR)
print('batch / patch:', config['data']['batch_size'], config['data']['patch_size'])
print('epochs:', config['training']['epochs'])
print('early stopping:', config['pruning'])

config: C:\model\configs\experiments\hemorrhage\H-S00.yaml
manifest: C:/Datasets/hemorrhage/v1/metadata/common/06_preprocessing_manifest.csv
data root: C:/Datasets/hemorrhage/v1/model_ready/segresnet
output: C:\model\runs\segresnet\H-S00
batch / patch: 1 [160, 160, 32]
epochs: 300
early stopping: {'enabled': True, 'monitor': 'composite', 'min_epochs': 50, 'patience': 30, 'min_delta': 0.001}


## 의존성, GPU 및 데이터 확인

In [2]:
import torch
import monai
from brainon_ai.models.hemorrhage.segresnet.data import load_manifest

assert torch.cuda.is_available(), 'CUDA를 사용할 수 없습니다.'
manifest = load_manifest(
    config['experiment']['manifest_path'],
    data_root=config['data']['root_dir'],
)
print('Python:', sys.executable)
print('PyTorch / MONAI:', torch.__version__, monai.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print(manifest.groupby('split').size())

Python: c:\model\.venv\Scripts\python.exe
PyTorch / MONAI: 2.13.0+cu130 1.6.0
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM (GB): 8.0
split
test           40
train         186
validation     40
dtype: int64


## 학습 시작

처음 시작할 때는 `training.resume: null`, 중단된 학습을 이어갈 때는 `training.resume: auto`로 바꿉니다. 기존 run이 있는데 `null`이면 덮어쓰지 않고 오류로 종료합니다.

In [3]:
import os
import subprocess

RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
child_env = os.environ.copy()
child_env['PYTHONPATH'] = str(REPO_ROOT / 'src') + os.pathsep + child_env.get('PYTHONPATH', '')
command = [
    sys.executable, '-m',
    'brainon_ai.models.hemorrhage.segresnet.train',
    '--config', str(CONFIG_PATH),
]
log_handle = LOG_PATH.open('a', encoding='utf-8')
process = subprocess.Popen(
    command, cwd=REPO_ROOT, env=child_env,
    stdout=log_handle, stderr=subprocess.STDOUT,
)
log_handle.close()
print('PID:', process.pid)
print('log:', LOG_PATH)
print('progress:', PROGRESS_PATH)
print('output:', RUN_DIR)

PID: 20204
log: C:\model\runs\segresnet\H-S00-launch.log
progress: C:\model\runs\segresnet\H-S00\progress.json
output: C:\model\runs\segresnet\H-S00


## 진행률 및 ETA

모니터링 셀을 중단해도 학습 프로세스는 계속 실행됩니다.

In [ ]:
import json
import time
from datetime import timedelta
from IPython.display import clear_output
from tqdm.auto import tqdm

terminal_states = {'completed', 'early_stopped', 'failed'}
bar = None
while True:
    if not PROGRESS_PATH.exists():
        clear_output(wait=True)
        print('trainer 초기화를 기다리는 중...')
        print('로그:', LOG_PATH)
        time.sleep(5)
        continue
    state = json.loads(PROGRESS_PATH.read_text(encoding='utf-8'))
    if bar is None:
        bar = tqdm(total=state['total_epochs'], initial=state['current_epoch'], desc='H-S00 epochs')
    bar.n = state['current_epoch']
    bar.set_postfix({
        'loss': state.get('train_loss'),
        'dice': state.get('dice'),
        'recall': state.get('recall'),
        'composite': state.get('composite'),
    })
    bar.refresh()
    eta = state.get('eta_seconds')
    print('status:', state['status'])
    print('epoch:', f"{state['current_epoch']}/{state['total_epochs']}")
    print('ETA:', str(timedelta(seconds=int(eta))) if eta is not None else '계산 중')
    print('best dice / recall / composite:', state.get('best_dice'), state.get('best_recall'), state.get('best_composite'))
    print('early stopping:', f"{state.get('pruning_bad_checks', 0)}/{state.get('pruning_patience', 0)} validation checks")
    print('GPU max allocated (GB):', state.get('gpu_max_memory_gb'))
    if state['status'] in terminal_states:
        bar.close()
        break
    time.sleep(10)

trainer 초기화를 기다리는 중...
로그: C:\model\runs\segresnet\H-S00-launch.log


H-S00 epochs:   0%|          | 0/300 [00:00<?, ?it/s]

status: running
epoch: 0/300
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30 validation checks
GPU max allocated (GB): 0.018
status: running
epoch: 0/300
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30 validation checks
GPU max allocated (GB): 0.018
status: running
epoch: 0/300
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30 validation checks
GPU max allocated (GB): 0.018
status: running
epoch: 0/300
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30 validation checks
GPU max allocated (GB): 0.018
status: running
epoch: 0/300
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30 validation checks
GPU max allocated (GB): 0.018
status: running
epoch: 0/300
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30 validation checks
GPU max allocated (GB): 0.018
status: running
epoch: 0/300
ETA: 계산 중
best dice / recall / composite:

## 최근 로그

In [ ]:
lines = LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()
print('\n'.join(lines[-80:]))